In [1]:
import pandas as pd

## 1. Dependence on $h$, $H$, $\mathcal{H}$

In [2]:
df = pd.read_csv("../results/experiment_cond_h.csv")

In [3]:
def format_value(v):
    if pd.isna(v):
        return ""
    elif isinstance(v, float):
        return f"{v:.6f}"
    elif isinstance(v, int):
        return f"{v:d}"
    else:
        return str(v)

In [4]:
def generate_table(_df, column, column_format):
    df = _df.copy()
    df["table entry"] = df.apply(lambda row: format_value(row[column]), axis=1)
    pivot_table = df.pivot_table(
        values="table entry", index="coarse m", columns="fine m", aggfunc="min"
    ).map(lambda x: "" if pd.isna(x) else x)
    pivot_table.columns.rename("$m \\rightarrow$", inplace=True)
    pivot_table.index.rename("$\\downarrow \\mathcal{M}$", inplace=True)
    pivot_table.rename(columns=lambda c: f"{{{c}}}", inplace=True)

    styles = pd.DataFrame("", index=pivot_table.index, columns=pivot_table.columns)
    checker_style = "background-color: #dddddd;"
    for i, idx in enumerate(pivot_table.index):
        for j, col in enumerate(pivot_table.columns):
            if i <= j and (i + j) % 2 == 0:
                styles.at[idx, col] = checker_style
    formatted_pivot_table = pivot_table.style.apply(lambda _df: styles, axis=None)
    formatted_pivot_table

    return (
        formatted_pivot_table.to_latex(
            convert_css=True,
            hrules=True,
            column_format="l|" + column_format * len(pivot_table.columns),
        )
        .replace("\\midrule", "\\specialrule{\\lightrulewidth}{0pt}{0pt}")
        .replace("\\bottomrule", "\\specialrule{\\heavyrulewidth}{0pt}{0pt}")
    )

In [5]:
col_format = "S[table-format=1.1e1, round-mode=figures, round-precision=2, exponent-mode=scientific, tight-spacing=true]"
cond_latex = generate_table(df, "cond", col_format)
with open("../docs/tables/experiment_cond_h.tex", "w") as f:
    f.write(cond_latex)

In [6]:
col_format = "S[table-format=4]"
iter_latex = generate_table(df, "iterations", col_format)
with open("../docs/tables/experiment_cond_h_iters.tex", "w") as f:
    f.write(iter_latex)

## 2. Dependence on $p$

In [7]:
df = pd.read_csv("../results/experiment_cond_p.csv")

In [8]:
summary = df[["polynomial degree", "iterations", "cond"]].copy()
summary = summary[summary["polynomial degree"] <= 12]  # Adjust as needed
summary["{$\\operatorname{cond}(T) / p^2$}"] = (
    summary["cond"] / summary["polynomial degree"] ** 2
)
summary.rename(
    columns={
        "polynomial degree": "{$p$}",
        "iterations": "{PCG iterations}",
        "cond": "{$\\operatorname{cond}(T)$}",
    },
    inplace=True,
)
summary

,{$p$},{PCG iterations},{$\operatorname{cond}(T)$},{$\operatorname{cond}(T) / p^2$}
0,1,21,12.269330,12.269330
1,2,42,61.014122,15.253530
2,3,64,143.400024,15.933336
3,4,86,260.149968,16.259373
4,5,105,410.441387,16.417655
5,6,126,595.079059,16.529974
6,7,149,813.860361,16.609395
7,8,168,1065.693645,16.651463
8,9,189,1351.682010,16.687432
9,10,222,1684.177577,16.841776


In [9]:
p_latex = summary.style.hide(axis="index").to_latex(
    hrules=True,
    column_format="S[table-format=2]"
    "S[table-format=3]"
    "S[table-format=1.2e2, round-mode=figures, round-precision=3, exponent-mode=scientific]"
    "S[table-format=2.2, round-mode=places, round-precision=2]",
)

In [10]:
with open("../docs/tables/experiment_cond_p.tex", "w") as f:
    f.write(p_latex)